# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates how to explore and analyze the [FAIR²](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset metadata and structure are provided via a Croissant schema URL:

```
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json
```

We will use Croissant's `@id` fields to reference all entities.

In [ ]:
# Ensure `mlcroissant` is installed and up-to-date
!pip install -U mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset 
dataset = mlc.Dataset(croissant_url)
# Print basic metadata (see note - we use attributes, not subscripting)
print(f"Dataset Name: {dataset.metadata.name}\n")
print(f"Description: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

Let's list all record sets and their main field and column IDs.

In [ ]:
# List available record sets in the Croissant metadata schema
print("Available Record Sets (referenced by their @id):\n")
for rs in dataset.metadata.record_sets:
    print(f"- Record Set Name: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Description: {rs.description if hasattr(rs, 'description') else 'N/A'}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.id}  (name: {field.name}, type: {field.data_type})")
    if rs.columns:
        print("  Columns:")
        for col in rs.columns:
            print(f"    - {col.id}  (name: {col.name}, type: {col.data_type})")
    print()

Below we inspect a preview of records for each record set using their `@id`.

In [ ]:
for rs in dataset.metadata.record_sets:
    print(f"=== RECORD SET: {rs.name} (@id: {rs.id}) ===")
    records_preview = []
    # Try to get up to the first 3 records
    try:
        for rec in dataset.records(record_set=rs.id):
            records_preview.append(rec)
            if len(records_preview) == 3:
                break
    except Exception as e:
        print("  Could not load records (possibly documentation/file-based record set). Error:", e)
        continue
    if records_preview:
        print(pd.DataFrame(records_preview).head(3))
    else:
        print("  No records previewable (possibly metadata-only set)")
    print("\n")

## 3. Data Extraction

Extract data from a record set using its `@id` into a pandas DataFrame for analysis.
Pick the record set containing the experimental results (often named something like "experiment" or containing the main table). Refer and use its `@id`.

In [ ]:
# Collect record set @ids
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}
for record_set_id in record_set_ids:
    # Use mlcroissant to read records as list of dict
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
    except Exception as e:
        print(f"Failed loading records for {record_set_id}: {e}")

# For main analysis select the main experimental record set (usually the largest DataFrame)
main_record_set = max(dataframes.items(), key=lambda x: x[1].shape[0])[0] if dataframes else None
print(f"Using record set @id: {main_record_set}")
print(f"Columns:\n{dataframes[main_record_set].columns.tolist()}")
dataframes[main_record_set].head()

## 4. Exploratory Data Analysis (EDA)

Apply typical data processing steps, such as filtering records on a numeric field, normalizing values, and grouping.

First, let's inspect the types of columns in our main record set and pick a numeric field by its `@id`.

In [ ]:
df = dataframes[main_record_set]
print("Numeric-type columns and their non-null counts:")
numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
for c in numeric_cols:
    print(f"- {c}: {df[c].notnull().sum()} non-missing values")

# For demo, pick first numeric column for filtering/analysis
numeric_field_id = numeric_cols[0] if numeric_cols else None
if numeric_field_id:
    threshold = df[numeric_field_id].quantile(0.5)  # median for demonstration
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"\nFiltered to records where {numeric_field_id} > {threshold:.3f} (median):\n")
    print(filtered_df.head())
    # Normalization (z-score)
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} (z-score):\n")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    # Try to group by a categorical field (if present)
    possible_group_fields = [c for c in df.columns if df[c].dtype == 'object' and df[c].nunique() < 10]
    if possible_group_fields:
        group_field_id = possible_group_fields[0]
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"\nGrouped mean by {group_field_id}:")
        print(grouped_df.head())
    else:
        print("\nNo suitable categorical field found for grouping.")
else:
    print("No numeric fields available for analysis.")

## 5. Visualization

Visualize the distribution of the main numeric field and its grouping if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id:
    plt.figure(figsize=(6,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.tight_layout() 
    plt.show()
    
    # If grouped field existed earlier, plot violin/box plot
    if 'group_field_id' in locals():
        plt.figure(figsize=(7,4))
        sns.boxplot(x=filtered_df[group_field_id], y=filtered_df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.tight_layout()
        plt.show()
else:
    print("Numeric field not available for visualization.")

## 6. Conclusion

- We loaded and reviewed the metadata and tabular content from the FAIR² dataset using its Croissant schema URL.
- Record sets and their fields were listed by `@id`, as per best practice for Croissant datasets.
- A main table was extracted for analysis, and we demonstrated filtering, normalization, grouping, and visualization operations.

**Next steps:** Deeper analysis, custom filtering, and domain-specific protein statistics can be performed using the DataFrame (see documented fields/columns above).